# 경로 확장 기법 (Path Expansion Trick) - 로짓의 경로 확장

- Tutorial ID: `tut-6-1`
- Tutorial: 경로 확장 기법 (Path Expansion Trick)
- Section ID: `s6-1-1`
- Section: 로짓의 경로 확장


In [ ]:
# ============================================================
# 코드 읽는 법 — 로짓의 경로 확장
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) logit이 확률분포로 바뀌는 과정과 temperature/top-k/top-p의 효과 관찰
#   3) 미래 토큰을 -inf로 막은 뒤 softmax 확률이 0이 되는지 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np

print("=" * 60)
print("경로 확장 기법 (Path Expansion Trick)")
print("=" * 60)

def softmax(x, axis=-1):
        x = x - np.max(x, axis=axis, keepdims=True)
    return np.exp(x) / np.sum(np.exp(x), axis=axis, keepdims=True)

# 설정
vocab_size = 5
d_model = 4
d_head = 2
seq_len = 3

vocab = ['A', 'B', 'C', 'D', 'E']

np.random.seed(10)

# 가중치
W_E = np.random.randn(vocab_size, d_model) * 0.4
W_U = np.random.randn(d_model, vocab_size) * 0.3

# 1-레이어 어텐션 헤드
W_Q = np.random.randn(d_model, d_head) * 0.3
W_K = np.random.randn(d_model, d_head) * 0.3
W_V = np.random.randn(d_model, d_head) * 0.3
W_O = np.random.randn(d_head, d_model) * 0.3

# 시퀀스: A B C → C 이후 예측
sequence = [0, 1, 2]  # A, B, C
X_embeds = W_E[sequence]  # (3, 4)

In [ ]:
# 경로별 기여 계산

In [ ]:
# 마지막 토큰 C에 대한 로짓 계산
last_embed = X_embeds[-1]  # C의 임베딩

In [ ]:
direct_logits = last_embed @ W_U
print("경로 1: 직접 경로 (바이그램)")
print(f"  C의 직접 로짓: {np.round(direct_logits, 3)}")
print(f"  가장 높은 토큰: {vocab[np.argmax(direct_logits)]}")

In [ ]:
Q = X_embeds @ W_Q  # (3, 2)
K = X_embeds @ W_K  # (3, 2)
V = X_embeds @ W_V  # (3, 2)

attn_scores = Q @ K.T / np.sqrt(d_head)
mask = np.triu(np.full((seq_len, seq_len), -1e9), k=1)
A = softmax(attn_scores + mask)  # (3, 3)

# 마지막 위치의 어텐션 출력
last_attn_weights = A[-1]  # C가 각 토큰에 주목하는 가중치
attn_result = last_attn_weights @ V  # (2,)
attn_logit_contribution = attn_result @ W_O @ W_U  # (5,)

print("
경로 2: 어텐션 헤드 기여")
print(f"  어텐션 가중치 (C가 A,B,C에): {np.round(last_attn_weights, 3)}")
print(f"  어텐션 헤드 로짓 기여: {np.round(attn_logit_contribution, 3)}")

# 개별 소스 토큰별 기여 분리
print("
  소스 토큰별 기여 분리:")
for i, (tok_id, weight) in enumerate(zip(sequence, last_attn_weights)):
    token_contrib = weight * V[i] @ W_O @ W_U
    max_idx = np.argmax(token_contrib)
    print(f"  {vocab[tok_id]} (가중치={weight:.3f}): "
          f"주로 '{vocab[max_idx]}' 예측 강화 ({token_contrib[max_idx]:.3f})")

In [ ]:
total_logits = direct_logits + attn_logit_contribution
total_probs = softmax(total_logits)

print("
" + "=" * 60)
print("최종 로짓 = 경로의 합")
print("=" * 60)
print(f"직접 경로: {np.round(direct_logits, 3)}")
print(f"어텐션:    {np.round(attn_logit_contribution, 3)}")
print(f"합계:      {np.round(total_logits, 3)}")
print(f"
예측 확률:")
for i, (tok, prob) in enumerate(zip(vocab, total_probs)):
    bar = '█' * int(prob * 30)
    print(f"  {tok}: {bar} {prob:.4f}")

In [ ]:
# 경로 확장의 중요성

In [ ]:
print("
" + "=" * 60)
print("경로 확장 기법의 중요성")
print("=" * 60)
print("""
전통적인 해석 방법의 한계:
  ❌ "레이어 3의 뉴런 427이 무엇을 하는가?"
  ❌ 잔차 스트림 벡터 자체를 분석 (의미 없음)

경로 확장의 장점:
  ✅ "A → 어텐션 헤드 h → 현재 위치에 B 기여"
  ✅ 어떤 경로가 중요한지 정량화 가능
  ✅ 가중치에서 직접 읽기 가능 (모델 실행 불필요)

2-레이어 경로 목록:
  1. 직접 경로: W_U W_E x
  2. Layer1 헤드 기여: W_U W_OV^{h1} A^{h1} W_E x
  3. Layer2 헤드 기여: W_U W_OV^{h2} A^{h2} (x + Σ_{h1} ...)
  4. V-합성 (가상 헤드): W_U W_OV^{h2} W_OV^{h1} A^{h2} A^{h1} W_E x
  5. QK 합성 (인덕션!): A^{h2}가 A^{h1}에 의존하는 경로

각 경로는 독립적으로 해석 및 검증 가능합니다.
""")